# 02. 데이터 탐색

국가별 장수, 예시 이미지 그리드, 혼동 쌍 비교, 촬영 위치 분포. 런타임: CPU로 충분하다.

In [ ]:
# 공통 셋업: 저장소 clone, 패키지 설치, Drive 연결
import os, sys
REPO_URL = "https://github.com/Cranezero/aix-assignment-pr.git"
REPO_DIR = "/content/aix-assignment-pr"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from src import config
from src.data import load_split, prepare_images
from src.utils import export_outputs, restore_outputs

restore_outputs()  # results/, assets/ 폴더 생성 + Drive에 저장된 이전 결과 가져오기

image_dir = prepare_images()
splits = {s: load_split(s) for s in ["train", "val", "test"]}
all_df = pd.concat([df.assign(split=s) for s, df in splits.items()], ignore_index=True)

## 국가별 장수

In [ ]:
table = all_df.pivot_table(index="country", columns="split", values="id", aggfunc="count")
table = table.loc[config.CODES, ["train", "val", "test"]]
table.insert(0, "name", [config.COUNTRIES[c] for c in table.index])
display(table)
print(table[["train", "val", "test"]].sum())

## 국가별 예시 이미지

In [ ]:
def crop_43(img):
    """그리드 표시용으로 가운데를 4:3으로 자른다."""
    w, h = img.size
    tw = min(w, round(h * 4 / 3))
    left = (w - tw) // 2
    return img.crop((left, 0, left + tw, h))

rng = np.random.default_rng(config.SEED)
train = splits["train"]
fig, axes = plt.subplots(4, 5, figsize=(15, 9.6))
for ax, code in zip(axes.flat, config.CODES):
    row = train[train["country"] == code].iloc[rng.integers(config.N_TRAIN)]
    ax.imshow(crop_43(Image.open(image_dir / f"{row['id']}.jpg")))
    ax.set_title(f"{config.COUNTRIES[code]} ({code})", fontsize=11)
    ax.axis("off")
plt.tight_layout()
fig.text(0.5, -0.01, "Source: OSV-5M (CC-BY-SA-4.0)", ha="center", va="top", fontsize=9)
plt.savefig(config.ASSETS_DIR / "eda_grid.png", dpi=120, bbox_inches="tight")
plt.show()

## 혼동 쌍 비교

비슷해 보일 것으로 예상한 쌍에서 국가당 3장씩.

In [ ]:
pairs = [("KR", "JP"), ("ES", "PT"), ("US", "CA"), ("NO", "SE"), ("AU", "ZA")]
fig, axes = plt.subplots(len(pairs), 6, figsize=(18, 2.7 * len(pairs)))
for r, (a, b) in enumerate(pairs):
    for j, code in enumerate([a, a, a, b, b, b]):
        sub = train[train["country"] == code]
        row = sub.iloc[rng.integers(len(sub))]
        ax = axes[r, j]
        ax.imshow(crop_43(Image.open(image_dir / f"{row['id']}.jpg")))
        ax.set_title(config.COUNTRIES[code], fontsize=10)
        ax.axis("off")
plt.tight_layout()
plt.savefig(config.ASSETS_DIR / "eda_pairs.png", dpi=100, bbox_inches="tight")
plt.show()

## 촬영 위치 분포

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
cmap = plt.get_cmap("tab20")
for i, code in enumerate(config.CODES):
    d = all_df[all_df["country"] == code]
    ax.scatter(d["longitude"], d["latitude"], s=2, color=cmap(i), label=code)
ax.set_xlim(-180, 180); ax.set_ylim(-60, 80)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(ncol=10, markerscale=5, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.22))
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(config.ASSETS_DIR / "sample_map.png", dpi=110, bbox_inches="tight")
plt.show()

## 촬영 연도

In [ ]:
year = pd.to_datetime(all_df["captured_at"], unit="ms").dt.year
display(year.value_counts().sort_index().to_frame("count").T)

## 사람이 쓰는 구분 단서

| 국가 | 문자 | 통행 방향 |
| --- | --- | --- |
| 한국 | 한글 | 우측 |
| 일본 | 한자·가나 | 좌측 |
| 태국 | 타이 문자 | 좌측 |
| 인도네시아 | 라틴 문자 | 좌측 |
| 인도 | 데바나가리 등, 라틴 문자 | 좌측 |
| 스페인·포르투갈·프랑스·독일·노르웨이·스웨덴 | 라틴 문자 (언어별 철자 차이) | 우측 |
| 영국 | 라틴 문자 | 좌측 |
| 러시아 | 키릴 문자 | 우측 |
| 미국·캐나다 | 라틴 문자 (캐나다 일부 프랑스어) | 우측 |
| 멕시코·브라질·아르헨티나 | 라틴 문자 (스페인어, 브라질은 포르투갈어) | 우측 |
| 호주·남아공 | 라틴 문자 | 좌측 |

OSV-5M 메타데이터의 `drive_side` 열은 영국을 우측통행(0)으로 표시하고 있어 이 표는 메타데이터 대신 실제 통행 방향을 적었다.

In [ ]:
export_outputs()